# 09 - Jembatan non-IID bertingkat (Paper 4) : cari TITIK PUTUS FedAvg

**Tujuan:** menautkan dua ekstrem yang sudah kita punya -- **homogen** (nb08, FL
bekerja) dan **heterogen lintas-dataset** (nb03, FL kolaps) -- dengan sebuah
**sumbu heterogenitas TERKENDALI**. Caranya: ambil **SATU dataset** (homogen di
sumber), lalu suntikkan non-IID label via **Dirichlet(alpha)** ke K klien, dan
turunkan alpha dari ~IID (100) ke ekstrem (0.01). Kita cari **pada alpha berapa
FedAvg mulai divergen** (titik putus).

**Beda dari nb05:** nb05 mem-partisi GABUNGAN CIC+UNSW (mencampur dua sumber
heterogen dulu), sehingga variabel alpha & heterogenitas-sumber tercampur -> hasil
tak konsisten. nb09 memakai **satu dataset per sweep** (CIC-only, UNSW-only),
sehingga alpha menjadi SATU-SATUNYA sumber heterogenitas -> diagnostik bersih.

**Nilai ilmiah:** mengubah 'FL gagal' menjadi kurva 'FL bertahan sampai alpha=X,
lalu runtuh' -> menjelaskan MENGAPA lintas-dataset (setara alpha sangat kecil +
skema beda) gagal. Jembatan antara nb08 dan nb03.

**Konsistensi:** reuse `../aws/fed_node.py` (MLP 128,64; FedAvg; z-score) IDENTIK
nb03/nb08. Output: `fed_out/noniid_bridge.csv` + `noniid_bridge_curve.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + impor fed_node

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, balanced_accuracy_score
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
if AWS_DIR not in sys.path: sys.path.insert(0, AWS_DIR)
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
# Hyperparameter IDENTIK nb03/nb08 (apple-to-apple).
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEED=42; K=4
ALPHAS=(100.0, 10.0, 1.0, 0.5, 0.1, 0.05, 0.01)  # IID -> ekstrem non-IID
print('fed_node loaded | LAYERS', fed.LAYERS, '| K =', K, '| alphas =', ALPHAS)

## 1. Muat partisi per-dataset (dari nb01)

Tiap sweep memakai SATU dataset: train utk partisi klien, test utk evaluasi.
Tidak ada pencampuran CIC<->UNSW (itu kasus heterogen nb03).

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
data={}
for ds,(ftr,fte) in {'CIC':('cic_train.npz','cic_test.npz'),
                     'UNSW':('unsw_train.npz','unsw_test.npz')}.items():
    Xtr,ytr=load(ftr); Xte,yte=load(fte)
    if Xtr is not None:
        data[ds]=(Xtr,ytr,Xte,yte)
        print(f'{ds}: train {Xtr.shape} pos {round(float(ytr.mean()),3)} | test {Xte.shape}')
ok=len(data)>0
if not ok: print('(partisi belum ada - jalankan nb01)')

## 2. Partisi Dirichlet(alpha) berbasis label + FedAvg

`alpha` besar (100) ~ IID; `alpha` kecil (0.01) = tiap klien condong satu kelas
(non-IID ekstrem). Dengan satu dataset, alpha = SATU-SATUNYA sumber heterogenitas.

In [ ]:
def dirichlet_partition(X, y, k, alpha, seed=SEED):
    r=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    for c in idx_by_c: r.shuffle(idx_by_c[c])
    client_idx=[[] for _ in range(k)]
    for c,idc in idx_by_c.items():
        prop=r.dirichlet([alpha]*k)
        cuts=(np.cumsum(prop)*len(idc)).astype(int)[:-1]
        for ki,part in enumerate(np.split(idc, cuts)):
            client_idx[ki].extend(part.tolist())
    parts=[]
    for ki in range(k):
        ii=np.array(client_idx[ki],int); r.shuffle(ii)
        if len(ii)>0: parts.append((X[ii], y[ii]))
    return parts
def run_fedavg(parts, Xte, yte, mu_g, sd_g, rounds=ROUNDS):
    zc=[fed.zscore_fit(X) for (X,y) in parts]
    Xz=[fed.zscore_apply(parts[i][0], *zc[i]) for i in range(len(parts))]
    sizes=[len(parts[i][1]) for i in range(len(parts))]
    Wg=fed.init_weights(SEED); curve=[]
    for t in range(rounds):
        cw=[]; cs=[]
        for i in range(len(parts)):
            Wl=fed.train_local(Wg, Xz[i], parts[i][1], epochs=LOCAL_EPOCHS, lr=LR,
                               mu=0.0, W_global=Wg, seed=SEED)
            cw.append(Wl); cs.append(sizes[i])
        Wg=fed.fedavg(cw, cs)
        out,_=fed.forward(Wg, fed.zscore_apply(Xte,mu_g,sd_g))
        curve.append(round(matthews_corrcoef(yte,(out>=0.5).astype(int)),4))
    return Wg, curve
def mcc_of(W, X, mu, sd, y):
    out,_=fed.forward(W, fed.zscore_apply(X,mu,sd)); return matthews_corrcoef(y,(out>=0.5).astype(int))
print('partisi Dirichlet + FedAvg siap')

## 3. Sweep alpha per-dataset + baseline centralized

Untuk tiap dataset: centralized (acuan) + FedAvg di tiap alpha. `label_skew` =
rata-rata |pos-rate klien - pos-rate global| sebagai proksi derajat non-IID.

In [ ]:
res=[]; curves=[]
if ok:
    for ds,(Xtr,ytr,Xte,yte) in data.items():
        mu_g,sd_g=fed.zscore_fit(Xtr)
        # centralized acuan
        Wc=fed.init_weights(SEED)
        Wc=fed.train_local(Wc, fed.zscore_apply(Xtr,mu_g,sd_g), ytr,
                           epochs=ROUNDS*LOCAL_EPOCHS, lr=LR, mu=0.0, seed=SEED)
        cen=round(mcc_of(Wc,Xte,mu_g,sd_g,yte),4)
        res.append({'dataset':ds,'alpha':'centralized','n_clients':1,'label_skew':0.0,'MCC':cen})
        print(f'[{ds}] centralized MCC={cen}')
        for alpha in ALPHAS:
            parts=dirichlet_partition(Xtr, ytr, K, alpha, seed=SEED)
            if len(parts)<2:
                res.append({'dataset':ds,'alpha':alpha,'n_clients':len(parts),'label_skew':None,'MCC':None,'note':'degenerate'}); continue
            skew=round(float(np.mean([abs(p[1].mean()-ytr.mean()) for p in parts])),4)
            Wg,curve=run_fedavg(parts, Xte, yte, mu_g, sd_g)
            mcc=round(mcc_of(Wg,Xte,mu_g,sd_g,yte),4)
            res.append({'dataset':ds,'alpha':alpha,'n_clients':len(parts),'label_skew':skew,'MCC':mcc})
            for t,v in enumerate(curve): curves.append({'dataset':ds,'alpha':alpha,'round':t,'MCC':v})
            print(f'[{ds}] alpha={alpha} skew={skew} MCC={mcc}')
    print('sweep selesai')
else:
    print('(lewati: partisi belum ada - jalankan nb01)')
print('=== SEL 3 (sweep alpha per-dataset) SELESAI ===')

## 4. Simpan + identifikasi titik putus

**Titik putus** = alpha terbesar di mana MCC federated jatuh di bawah ambang
(mis. < 50% MCC centralized). Dilaporkan apa adanya per dataset.

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'noniid_bridge.csv'),index=False)
    pd.DataFrame(curves).to_csv(os.path.join(OUTDIR,'noniid_bridge_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    # ringkas titik putus (ambang 50% centralized)
    for ds in dfres['dataset'].unique():
        sub=dfres[(dfres['dataset']==ds)]
        cen=sub[sub['alpha']=='centralized']['MCC']
        if len(cen)==0 or cen.iloc[0] is None: continue
        thr=0.5*float(cen.iloc[0])
        fed_rows=sub[sub['alpha']!='centralized'].copy()
        fed_rows=fed_rows[fed_rows['MCC'].notna()]
        broke=fed_rows[fed_rows['MCC']<thr]
        bp=broke['alpha'].max() if len(broke) else 'tak putus pd rentang alpha'
        print(f'[{ds}] centralized={cen.iloc[0]} ambang(50%)={round(thr,4)} -> titik putus alpha~{bp}')
    print('\nInterpretasi: makin kecil alpha (makin non-IID) MCC turun; titik putus = batas FedAvg naif.')
else:
    print('(tak ada hasil - jalankan nb01 dulu)')
print('=== SEL 4 (titik putus) SELESAI ===')

## 5. Unggah hasil ke S3 (results/noniid_bridge)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1')
PREFIX='unsw-far/federated/results/noniid_bridge'
try:
    import boto3
    s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('noniid_bridge.csv','noniid_bridge_curve.csv'):
        p=os.path.join(OUTDIR,f)
        if os.path.exists(p):
            s3.upload_file(p, S3_BUCKET, f'{PREFIX}/{f}'); up+=1; print('  ok ->', f)
    print(f'\nunggah {up} berkas -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e:
    print('(upload S3 gagal:',e,')')
print('=== SEL 5 (unggah S3) SELESAI ===')

## 6. Catatan untuk naskah

- Hasil = **jembatan** antara nb08 (homogen, FL bekerja) dan nb03 (heterogen
  lintas-dataset, FL kolaps). Memberi kurva MCC-vs-alpha + titik putus.
- Karena satu dataset per sweep, **alpha = satu-satunya sumber heterogenitas**
  (bersih; beda dari nb05 yang mencampur sumber).
- Narasi Paper 4: FedAvg naif bertahan sampai derajat non-IID tertentu, lalu
  runtuh -> lintas-dataset (skema+label beda) berada DI LUAR batas itu ->
  memotivasi mitigasi (FedProx mu besar, FedBN/personalized, atau one-class).
- Logika FL identik `aws/fed_node.py` -> apple-to-apple dgn nb03/nb08.